# HMM for Noun Phrase (NP) Chunking
Tags: `B-NP`, `I-NP`, `O`. Tasks: (1) Forward algorithm for the probability of each test sentence, (2) Viterbi decoding + evaluation with `seqeval`. Add-K smoothing is used for zero counts.

In [1]:
!pip install seqeval

In [2]:
import numpy as np
from collections import defaultdict
from seqeval.metrics import classification_report, f1_score, precision_score, recall_score, accuracy_score

## 1. Load data
Each line is `word<TAB>tag`; blank lines separate sentences.

In [3]:
def read_data(path):
    sentences, tags = [], []
    words, labs = [], []
    with open(path, encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                if words:
                    sentences.append(words); tags.append(labs)
                    words, labs = [], []
            else:
                w, t = line.split("\t")
                words.append(w); labs.append(t)
    if words:
        sentences.append(words); tags.append(labs)
    return sentences, tags

train_sents, train_tags = read_data("train_NP.txt")
test_sents, test_tags = read_data("test_NP.txt")
print("Train sentences:", len(train_sents), "| Test sentences:", len(test_sents))

Train sentences: 823 | Test sentences: 77


## 2. Estimate HMM parameters (with Add-K smoothing)
- Initial: π(t) = (C(start→t)+K) / (N_sent + K·|T|)
- Transition: A(t'|t) = (C(t→t')+K) / (C(t)+K·|T|)
- Emission: B(w|t) = (C(t,w)+K) / (C(t)+K·(|V|+1)), where the extra `<UNK>` slot handles unseen test words.

In [4]:
K = 0.01
tags = ["B-NP", "I-NP", "O"]
tag2id = {t: i for i, t in enumerate(tags)}
T = len(tags)

vocab = sorted({w for s in train_sents for w in s})
word2id = {w: i for i, w in enumerate(vocab)}
UNK = len(vocab)          # index reserved for unknown words
V = len(vocab) + 1

init_c = np.zeros(T)
trans_c = np.zeros((T, T))
emit_c = np.zeros((T, V))
for s, t in zip(train_sents, train_tags):
    ids = [tag2id[x] for x in t]
    init_c[ids[0]] += 1
    for i, (w, ti) in enumerate(zip(s, ids)):
        emit_c[ti, word2id[w]] += 1
        if i > 0:
            trans_c[ids[i-1], ti] += 1

pi = (init_c + K) / (init_c.sum() + K * T)
A = (trans_c + K) / (trans_c.sum(axis=1, keepdims=True) + K * T)
B = (emit_c + K) / (emit_c.sum(axis=1, keepdims=True) + K * V)

log_pi, log_A, log_B = np.log(pi), np.log(A), np.log(B)
print("Initial probs:", dict(zip(tags, pi.round(4))))
print("Transition matrix:\n", A.round(4))

Initial probs: {'B-NP': np.float64(0.6573), 'I-NP': np.float64(0.0), 'O': np.float64(0.3426)}
Transition matrix:
 [[0.0295 0.6956 0.2749]
 [0.0459 0.3875 0.5666]
 [0.5403 0.     0.4597]]


## 3. Forward algorithm
Computes P(sentence) = Σ over all tag sequences of P(words, tags). Computed in log space (with log-sum-exp) to avoid underflow.

In [5]:
def obs_ids(sentence):
    return [word2id.get(w, UNK) for w in sentence]

def forward_log_prob(sentence):
    o = obs_ids(sentence)
    n = len(o)
    alpha = np.zeros((n, T))
    alpha[0] = log_pi + log_B[:, o[0]]
    for t in range(1, n):
        for j in range(T):
            alpha[t, j] = np.logaddexp.reduce(alpha[t-1] + log_A[:, j]) + log_B[j, o[t]]
    return np.logaddexp.reduce(alpha[-1])

forward_logp = [forward_log_prob(s) for s in test_sents]
for i in range(len(test_sents)):
    print(f"Sentence {i+1}: log P = {forward_logp[i]:.4f}, P = {np.exp(forward_logp[i]):.4e}")

Sentence 1: log P = -289.3030, P = 2.2766e-126
Sentence 2: log P = -217.3809, P = 3.9144e-95
Sentence 3: log P = -238.7652, P = 2.0212e-104
Sentence 4: log P = -250.9309, P = 1.0522e-109
Sentence 5: log P = -248.9475, P = 7.6470e-109
Sentence 6: log P = -276.8823, P = 5.6434e-121
Sentence 7: log P = -286.4403, P = 3.9860e-125
Sentence 8: log P = -201.1129, P = 4.5476e-88
Sentence 9: log P = -227.7439, P = 1.2362e-99
Sentence 10: log P = -188.2185, P = 1.8103e-82
Sentence 11: log P = -124.0753, P = 1.3025e-54
Sentence 12: log P = -180.0706, P = 6.2565e-79
Sentence 13: log P = -202.4967, P = 1.1398e-88
Sentence 14: log P = -367.2369, P = 3.2436e-160
Sentence 15: log P = -166.5274, P = 4.7653e-73
Sentence 16: log P = -87.4052, P = 1.0975e-38
Sentence 17: log P = -229.4334, P = 2.2821e-100
Sentence 18: log P = -89.9215, P = 8.8629e-40
Sentence 19: log P = -155.3773, P = 3.3152e-68
Sentence 20: log P = -147.9533, P = 5.5552e-65
Sentence 21: log P = -150.2163, P = 5.7796e-66
Sentence 22: log

## 4. Viterbi decoding

In [6]:
def viterbi(sentence):
    o = obs_ids(sentence)
    n = len(o)
    delta = np.zeros((n, T))
    psi = np.zeros((n, T), dtype=int)
    delta[0] = log_pi + log_B[:, o[0]]
    for t in range(1, n):
        for j in range(T):
            scores = delta[t-1] + log_A[:, j]
            psi[t, j] = np.argmax(scores)
            delta[t, j] = scores[psi[t, j]] + log_B[j, o[t]]
    path = [int(np.argmax(delta[-1]))]
    for t in range(n-1, 0, -1):
        path.append(int(psi[t, path[-1]]))
    path.reverse()
    return [tags[i] for i in path]

pred_tags = [viterbi(s) for s in test_sents]
for w, p, g in list(zip(test_sents, pred_tags, test_tags))[:3]:
    print(list(zip(w, g, p)), "\n")

[('Confidence', 'B-NP', 'B-NP'), ('in', 'O', 'O'), ('the', 'B-NP', 'B-NP'), ('pound', 'I-NP', 'I-NP'), ('is', 'O', 'O'), ('widely', 'O', 'B-NP'), ('expected', 'O', 'O'), ('to', 'O', 'O'), ('take', 'O', 'O'), ('another', 'B-NP', 'B-NP'), ('sharp', 'I-NP', 'I-NP'), ('dive', 'I-NP', 'I-NP'), ('if', 'O', 'O'), ('trade', 'B-NP', 'B-NP'), ('figures', 'I-NP', 'I-NP'), ('for', 'O', 'O'), ('September', 'B-NP', 'B-NP'), (',', 'O', 'O'), ('due', 'O', 'O'), ('for', 'O', 'O'), ('release', 'B-NP', 'B-NP'), ('tomorrow', 'B-NP', 'I-NP'), (',', 'O', 'O'), ('fail', 'O', 'B-NP'), ('to', 'O', 'O'), ('show', 'O', 'O'), ('a', 'B-NP', 'B-NP'), ('substantial', 'I-NP', 'B-NP'), ('improvement', 'I-NP', 'I-NP'), ('from', 'O', 'O'), ('July', 'B-NP', 'B-NP'), ('and', 'I-NP', 'I-NP'), ('August', 'I-NP', 'I-NP'), ("'s", 'B-NP', 'B-NP'), ('near-record', 'I-NP', 'I-NP'), ('deficits', 'I-NP', 'I-NP'), ('.', 'O', 'O')] 

[('Chancellor', 'O', 'B-NP'), ('of', 'O', 'O'), ('the', 'B-NP', 'B-NP'), ('Exchequer', 'I-NP', 'I-NP

## 5. Evaluation with seqeval

In [7]:
print(classification_report(test_tags, pred_tags))
print("Precision:", precision_score(test_tags, pred_tags))
print("Recall   :", recall_score(test_tags, pred_tags))
print("F1       :", f1_score(test_tags, pred_tags))
print("Accuracy :", accuracy_score(test_tags, pred_tags))

              precision    recall  f1-score   support

          NP       0.64      0.72      0.68       475

   micro avg       0.64      0.72      0.68       475
   macro avg       0.64      0.72      0.68       475
weighted avg       0.64      0.72      0.68       475

Precision: 0.6380597014925373
Recall   : 0.72
F1       : 0.6765578635014836
Accuracy : 0.8523206751054853
